# ⚙️ Activity 05: Linear Regression From Scratch

**Track:** Linear Regression  
**Level:** Advanced  

---

## 📖 What You Will Learn
- Derive and implement the **OLS closed-form** solution
- Implement **Batch Gradient Descent** from scratch
- Implement **Stochastic & Mini-Batch Gradient Descent**
- Track and visualise the **loss curve**
- Compare your implementation against sklearn
- Debug gradient descent (learning rate sensitivity)

---

## 🧠 Concept: The Math

**Model:**
$$\hat{y} = X\beta$$

**Loss (MSE):**
$$\mathcal{L}(\beta) = \frac{1}{n}\|y - X\beta\|^2$$

**Gradient:**
$$\frac{\partial \mathcal{L}}{\partial \beta} = -\frac{2}{n}X^T(y - X\beta)$$

**Gradient Descent Update:**
$$\beta \leftarrow \beta - \alpha \cdot \frac{\partial \mathcal{L}}{\partial \beta}$$

**OLS Closed-form:**
$$\hat{\beta} = (X^TX)^{-1}X^Ty$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression as SklearnLinReg
from sklearn.metrics import r2_score, mean_squared_error

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Ready')

## 📊 Data Preparation

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

X_raw = df.drop(columns=['MedHouseVal']).values
y_raw = df['MedHouseVal'].values

# Scale (important for gradient descent convergence!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_raw, test_size=0.2, random_state=42)

# Add bias column (column of 1s) for OLS and GD implementations
X_tr_b = np.column_stack([np.ones(len(X_tr)), X_tr])   # (n, p+1)
X_te_b = np.column_stack([np.ones(len(X_te)), X_te])

print(f'Train: {X_tr_b.shape}  Test: {X_te_b.shape}')

## 🔢 Implementation 1 — OLS Closed-Form

In [ ]:
class OLSLinearRegression:
    """
    Linear Regression via the Normal Equation (closed-form OLS).
    β = (XᵀX)⁻¹Xᵀy

    WHY use pinv instead of direct inverse?
    np.linalg.inv raises an error if XᵀX is singular (e.g., collinear features).
    np.linalg.pinv (pseudo-inverse) handles singular matrices gracefully.
    """
    def __init__(self):
        self.beta_ = None

    def fit(self, X, y):
        # COMMON ERROR: forgetting to add bias column before calling fit
        XtX  = X.T @ X
        Xty  = X.T @ y
        self.beta_ = np.linalg.pinv(XtX) @ Xty
        return self

    def predict(self, X):
        return X @ self.beta_

    def score(self, X, y):
        return r2_score(y, self.predict(X))


ols = OLSLinearRegression()
ols.fit(X_tr_b, y_tr)

r2_ols = ols.score(X_te_b, y_te)
rmse_ols = np.sqrt(mean_squared_error(y_te, ols.predict(X_te_b)))
print(f'OLS (from scratch) — R²: {r2_ols:.4f}  RMSE: {rmse_ols:.4f}')

## 🏃 Implementation 2 — Batch Gradient Descent

In [ ]:
class GradientDescentLinearRegression:
    """
    Linear Regression via Batch Gradient Descent.

    Each step uses ALL training samples to compute the gradient.
    This is stable but slow for very large datasets.
    """
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr  = learning_rate
        self.n   = n_iterations
        self.beta_ = None
        self.loss_history_ = []

    def _mse(self, X, y):
        return np.mean((y - X @ self.beta_) ** 2)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.beta_ = np.zeros(n_features)  # initialise weights to zero

        for i in range(self.n):
            # Compute prediction errors
            residuals = y - X @ self.beta_

            # Gradient of MSE with respect to beta
            # ∂L/∂β = -2/n * Xᵀ(y - Xβ)
            gradient = -2 / n_samples * X.T @ residuals

            # Update rule
            self.beta_ -= self.lr * gradient

            # Track loss every 50 iterations
            if i % 50 == 0:
                self.loss_history_.append(self._mse(X, y))

        return self

    def predict(self, X):
        return X @ self.beta_

    def score(self, X, y):
        return r2_score(y, self.predict(X))


gd = GradientDescentLinearRegression(learning_rate=0.05, n_iterations=2000)
gd.fit(X_tr_b, y_tr)

r2_gd = gd.score(X_te_b, y_te)
print(f'Batch GD (from scratch) — R²: {r2_gd:.4f}')

# Plot loss curve
plt.figure(figsize=(9, 4))
plt.plot(gd.loss_history_, color='steelblue')
plt.xlabel('Iteration (×50)')
plt.ylabel('MSE')
plt.title('Gradient Descent — Loss Curve')
plt.yscale('log')
plt.tight_layout()
plt.show()

## ⚡ Implementation 3 — Mini-Batch Gradient Descent

### 🧠 Concept: Variants of Gradient Descent

| Variant | Batch size | Stability | Speed |
|---|---|---|---|
| Batch GD | All n samples | High | Slow (large n) |
| Stochastic GD (SGD) | 1 sample | Noisy | Fast |
| Mini-Batch GD | 32–256 | Balanced | Best in practice |

> **Deep learning uses mini-batch GD** exclusively. SGD is misleadingly named — PyTorch `torch.optim.SGD` is actually mini-batch.

In [ ]:
class MiniBatchGDLinReg:
    """
    Linear Regression via Mini-Batch Gradient Descent.
    Best balance between stability and computational efficiency.
    """
    def __init__(self, lr=0.05, epochs=100, batch_size=64):
        self.lr         = lr
        self.epochs     = epochs
        self.batch_size = batch_size
        self.beta_      = None
        self.loss_history_ = []

    def fit(self, X, y):
        n, p = X.shape
        self.beta_ = np.zeros(p)

        for epoch in range(self.epochs):
            # Shuffle at each epoch
            idx = np.random.permutation(n)
            X_shuffled = X[idx]
            y_shuffled = y[idx]

            epoch_loss = []
            for start in range(0, n, self.batch_size):
                Xb = X_shuffled[start:start+self.batch_size]
                yb = y_shuffled[start:start+self.batch_size]
                nb = len(Xb)

                residuals = yb - Xb @ self.beta_
                gradient  = -2/nb * Xb.T @ residuals
                self.beta_ -= self.lr * gradient
                epoch_loss.append(np.mean(residuals**2))

            self.loss_history_.append(np.mean(epoch_loss))

        return self

    def predict(self, X):
        return X @ self.beta_

    def score(self, X, y):
        return r2_score(y, self.predict(X))


sgd = MiniBatchGDLinReg(lr=0.05, epochs=200, batch_size=64)
sgd.fit(X_tr_b, y_tr)

r2_sgd = sgd.score(X_te_b, y_te)
print(f'Mini-Batch GD (from scratch) — R²: {r2_sgd:.4f}')

## 🔍 Step 4 — Learning Rate Sensitivity

In [ ]:
# ── DEBUG TIP: Too large LR → diverge; too small → very slow ─────────────────
lrs = [0.001, 0.01, 0.05, 0.3, 1.0]
plt.figure(figsize=(12, 5))

for lr in lrs:
    model = GradientDescentLinearRegression(learning_rate=lr, n_iterations=500)
    model.fit(X_tr_b, y_tr)
    # Clip for visualisation (divergent runs go to inf)
    losses = np.clip(model.loss_history_, 0, 200)
    plt.plot(losses, label=f'lr = {lr}')

plt.xlabel('Iteration (×50)')
plt.ylabel('MSE (clipped at 200)')
plt.title('Learning Rate Sensitivity')
plt.legend()
plt.tight_layout()
plt.show()

print('\n💡 lr=1.0 likely diverges (loss explodes).')
print('   lr=0.001 converges but slowly.')
print('   lr=0.05 is a good balance for this problem.')

## 📊 Step 5 — Compare All Implementations

In [ ]:
# sklearn reference
sklearn_lr = SklearnLinReg()
sklearn_lr.fit(X_tr, y_tr)
r2_sklearn = r2_score(y_te, sklearn_lr.predict(X_te))

summary = pd.DataFrame([
    {'Method': 'sklearn LinearRegression', 'R²': r2_sklearn},
    {'Method': 'OLS (Normal Equation)',    'R²': r2_ols},
    {'Method': 'Batch GD (from scratch)',  'R²': r2_gd},
    {'Method': 'Mini-Batch GD',            'R²': r2_sgd},
])

print(summary.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(summary['Method'], summary['R²'], color=['tomato','steelblue','limegreen','orange'])
plt.xlim(0, 1)
plt.xlabel('R²')
plt.title('R² Comparison: sklearn vs Scratch Implementations')
plt.tight_layout()
plt.show()

## 🎯 Summary

| Implementation | Key Insight |
|---|---|
| OLS (Normal Equation) | Exact, but O(p³) — slow for many features |
| Batch GD | Stable convergence; entire dataset per step |
| Mini-Batch GD | Best practice for large datasets / deep learning |
| sklearn | Uses optimised LAPACK under the hood (same as OLS) |

---

**Next:** [06_regularization_ridge_lasso.ipynb](06_regularization_ridge_lasso.ipynb)